[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C13_RL_Foundations_Course/03_policy_gradient/03_policy_gradient.ipynb)

# 03 · 策略梯度与 REINFORCE（纯 numpy）

目标：从零实现 **softmax 策略**、**score function**、**REINFORCE**、**baseline**、**熵正则**，在 toy chain 上收敛，并**数值验证**：baseline 不引入偏差（E[score]=0 到机器精度）却大幅降方差。

路线：softmax 策略 + score(对拍有限差分) → REINFORCE 收敛 → 因果性(只用 G_t) → baseline 无偏(E[score]=0) → baseline 降方差(~18×) → 熵正则 → ✏️ 练习 → 📖 答案 → 🧪 RLHF 风味胶囊。

> 心智模型：**θ ← θ + η·G·∇log π**：好动作(高回报)就推高其概率，坏动作就压低。baseline 把「绝对回报」换成「比平均好多少」(advantage)，信号更干净。

## 1 · softmax 策略与 score function（对拍有限差分）

表格 softmax 策略：每状态一组 logits $\theta_s$，$\pi(a|s) = \text{softmax}(\theta_s)_a$。

**score function** $\nabla_\theta \log\pi(a|s)$ 对 softmax 有闭式：$\frac{\partial \log\pi(a)}{\partial \theta_j} = \mathbb{1}[j=a] - \pi(j)$。这是策略梯度的原子，必须先验证它正确——用有限差分对拍。

In [ ]:
import numpy as np

def softmax(logits):
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()

def score(theta_s, a):
    '''∇_θ log π(a|s) = onehot(a) − π，对单状态 logits θ_s。'''
    pi = softmax(theta_s)
    g = -pi.copy()
    g[a] += 1.0
    return g

# 对拍有限差分
theta_s = np.array([0.5, -0.3, 0.1])
a = 1
g_analytic = score(theta_s, a)
eps = 1e-6
g_fd = np.zeros(3)
for j in range(3):
    tp = theta_s.copy(); tp[j] += eps
    tm = theta_s.copy(); tm[j] -= eps
    g_fd[j] = (np.log(softmax(tp)[a]) - np.log(softmax(tm)[a])) / (2 * eps)
print('score 解析   :', np.round(g_analytic, 6))
print('score 有限差分:', np.round(g_fd, 6))
assert np.allclose(g_analytic, g_fd, atol=1e-5), 'score function 必须对拍有限差分'
print('✅ score function ∇log π = onehot(a) − π 正确（对拍数值微分）')

## 2 · toy chain 环境 + 折扣回报

链 MDP：状态 `0..N-1` 排成一线，起点 0，目标 `N-1`(+1, 终止)，每步 −0.05，动作 `0=左, 1=右`(确定性)。

策略要学会「一路向右」。先写环境和折扣回报计算。

In [ ]:
class Chain:
    def __init__(self, N=5, gamma=0.99, step_reward=-0.05):
        self.N = N; self.gamma = gamma; self.step_reward = step_reward
        self.start = 0; self.goal = N - 1
    def reset(self):
        self.pos = self.start; return self.pos
    def step(self, a):
        self.pos = min(self.pos + 1, self.N - 1) if a == 1 else max(self.pos - 1, 0)
        done = (self.pos == self.goal)
        r = 1.0 if done else self.step_reward
        return self.pos, r, done

def discounted_returns(rewards, gamma):
    G = np.zeros(len(rewards)); run = 0.0
    for t in reversed(range(len(rewards))):
        run = rewards[t] + gamma * run
        G[t] = run
    return G

def rollout(env, theta, rng, max_steps=50):
    '''用当前 softmax 策略采一条轨迹，返回 states, actions, rewards。'''
    s = env.reset(); S, A, Rw = [], [], []
    for _ in range(max_steps):
        pi = softmax(theta[s])
        a = int(rng.choice(len(pi), p=pi))
        sp, r, done = env.step(a)
        S.append(s); A.append(a); Rw.append(r); s = sp
        if done: break
    return S, A, Rw

env = Chain(N=5, gamma=0.99)
rng = np.random.default_rng(0)
theta0 = np.zeros((env.N, 2))    # 初始均匀策略
S, A, Rw = rollout(env, theta0, rng)
G = discounted_returns(Rw, env.gamma)
print(f'一条轨迹长度 {len(S)}, 总奖励 {sum(Rw):.3f}')
print('回报 G_t:', np.round(G, 3))
# 回报应随接近终点递增(后面的步离 +1 更近)
assert G[-1] >= G[0] or len(S) <= 2, '靠近目标的步回报通常更高'
print('✅ chain 环境与折扣回报就绪')

## 3 · REINFORCE：蒙特卡洛策略梯度

更新 $\theta_s \leftarrow \theta_s + \eta\, \gamma^t\, G_t\, \nabla_\theta \log\pi(a_t|s_t)$。

训练后策略应在每个状态偏好「向右(1)」，回报曲线上升。

In [ ]:
def reinforce(env, n_episodes=3000, lr=0.2, seed=0, max_steps=50,
              baseline=False, normalize=False):
    rng = np.random.default_rng(seed)
    theta = np.zeros((env.N, 2))
    returns_hist = []
    Vbar = np.zeros(env.N)        # 运行 baseline（每状态平均回报）
    for ep in range(n_episodes):
        S, A, Rw = rollout(env, theta, rng, max_steps)
        G = discounted_returns(Rw, env.gamma)
        returns_hist.append(sum(Rw))
        if normalize and len(G) > 1:
            adv = (G - G.mean()) / (G.std() + 1e-8)
        elif baseline:
            adv = G - Vbar[S]
            for i, st in enumerate(S):
                Vbar[st] += 0.05 * (G[i] - Vbar[st])   # 在线更新 baseline
        else:
            adv = G
        for t, (st, at) in enumerate(zip(S, A)):
            theta[st] += lr * (env.gamma ** t) * adv[t] * score(theta[st], at)
    return theta, np.array(returns_hist)

theta_pg, ret_pg = reinforce(env, n_episodes=3000, lr=0.2, seed=0)
print(f'REINFORCE 末期平均总奖励 = {ret_pg[-200:].mean():.3f} (初期 {ret_pg[:200].mean():.3f})')
policy = [int(softmax(theta_pg[s]).argmax()) for s in range(env.N)]
probs_right = [softmax(theta_pg[s])[1] for s in range(env.N - 1)]
print('学到的策略(0=左,1=右):', policy[:-1], '| P(右):', np.round(probs_right, 3))
# 每个非终止状态都应偏好向右(朝目标)
assert all(p == 1 for p in policy[:-1]), '每个状态都应学会向右'
assert ret_pg[-200:].mean() > ret_pg[:200].mean(), '回报应随训练上升'
print('✅ REINFORCE 收敛：策略学会一路向右，回报上升到接近最优')

## 4 · 因果性：为什么用 $G_t$ 而非整条轨迹回报

策略梯度定理用从 $t$ 起的回报 $G_t$(reward-to-go)，而非整条轨迹回报 $R(\tau)$。原因：时刻 $t$ 的动作**只能影响 $t$ 之后**的奖励；把 $t$ 之前的奖励算进去**不改变梯度期望(无偏)**，却**徒增方差**。

为了让方差差异显现，用一个**每步带独立噪声**的链(之前确定性链把噪声全压在终点、看不出差别)。验证两点：① 两种权重的梯度**均值一致**(无偏)；② $G_t$ 的方差**更小**。

In [ ]:
class NoisyChain:
    '''每步奖励含独立高斯噪声 —— 这样早期步的「过去奖励」真的带噪，因果性的方差收益才显现。'''
    def __init__(self, N=6, gamma=0.99, step_reward=-0.05, noise=1.0):
        self.N = N; self.gamma = gamma; self.step_reward = step_reward; self.noise = noise
        self.start = 0; self.goal = N - 1
    def reset(self, rng): self.pos = self.start; self.rng = rng; return self.pos
    def step(self, a):
        self.pos = min(self.pos + 1, self.N - 1) if a == 1 else max(self.pos - 1, 0)
        done = (self.pos == self.goal)
        r = (1.0 if done else self.step_reward) + self.rng.normal(0, self.noise)  # 每步噪声
        return self.pos, r, done

def rollout_noisy(env, theta, rng, max_steps=50):
    s = env.reset(rng); S, A, Rw = [], [], []
    for _ in range(max_steps):
        a = int(rng.choice(2, p=softmax(theta[s])))
        sp, r, done = env.step(a); S.append(s); A.append(a); Rw.append(r); s = sp
        if done: break
    return S, A, Rw

def grad_estimator(env, theta, n_samples, rng, use_full_return=False):
    grads = []
    for _ in range(n_samples):
        S, A, Rw = rollout_noisy(env, theta, rng)
        G = discounted_returns(Rw, env.gamma)
        full = sum(Rw)                       # 整条轨迹总回报
        g = np.zeros_like(theta)
        for t, (st, at) in enumerate(zip(S, A)):
            weight = full if use_full_return else G[t]
            g[st] += weight * score(theta[st], at)
        grads.append(g.flatten())
    return np.array(grads)

envn = NoisyChain(N=6, noise=1.0)
theta_n = np.zeros((envn.N, 2)); theta_n[:, 1] = 1.0   # 固定策略(偏向右,轨迹较短)
g_Gt   = grad_estimator(envn, theta_n, 12000, np.random.default_rng(10), use_full_return=False)
g_full = grad_estimator(envn, theta_n, 12000, np.random.default_rng(10), use_full_return=True)
print('用 G_t   : 梯度总方差 =', round(g_Gt.var(0).sum(), 3))
print('用整轨回报: 梯度总方差 =', round(g_full.var(0).sum(), 3))
print('两者梯度均值最大差 =', round(np.max(np.abs(g_Gt.mean(0) - g_full.mean(0))), 4), '(应≈0，无偏)')
# ① 无偏：两种权重梯度均值一致(差异仅来自有限采样)
assert np.max(np.abs(g_Gt.mean(0) - g_full.mean(0))) < 0.05, 'reward-to-go 不改变梯度期望(无偏)'
# ② 降方差：G_t 方差更小
assert g_Gt.var(0).sum() < g_full.var(0).sum(), 'reward-to-go(G_t) 应比整轨回报方差更小'
print('✅ 因果性：丢掉过去奖励不改变梯度期望(无偏)，却降低方差 —— 免费的方差缩减')

## 5 · baseline 无偏的根源：E[score] = 0（机器精度）

baseline 不引入偏差，根源是 **score function 在策略下期望为零**：$\mathbb{E}_{a\sim\pi}[\nabla_\theta \log\pi(a|s)] = 0$。

因此 $\mathbb{E}[\nabla\log\pi \cdot b] = b\cdot\mathbb{E}[\nabla\log\pi] = 0$，且 $\mathbb{E}[\nabla\log\pi(G-b)] = \mathbb{E}[\nabla\log\pi\, G]$。我们**解析地**(对所有动作加权求和)验证到机器精度。

In [ ]:
theta_s = np.array([0.5, -0.3, 0.1, 0.8])
pi = softmax(theta_s)
# E_π[score] = Σ_a π(a) score(a)  —— 解析期望(非采样)
E_score = sum(pi[a] * score(theta_s, a) for a in range(4))
print('E_π[score] =', E_score)
print('max|E_π[score]| =', np.max(np.abs(E_score)))
assert np.allclose(E_score, 0.0, atol=1e-12), 'score 期望必须为 0(机器精度)'

# 推论1: 任意常数 baseline b 不改变期望
b = 7.3
E_score_b = sum(pi[a] * (score(theta_s, a) * b) for a in range(4))
assert np.allclose(E_score_b, 0.0, atol=1e-10), 'E[score·b]=0'

# 推论2: E[score·Q] == E[score·(Q−b)]  baseline 保持梯度不变
Q = np.array([1.0, 2.0, -1.0, 3.0])
E_sQ  = sum(pi[a] * score(theta_s, a) * Q[a] for a in range(4))
E_sQb = sum(pi[a] * score(theta_s, a) * (Q[a] - b) for a in range(4))
assert np.allclose(E_sQ, E_sQb, atol=1e-10), 'baseline 必须保持梯度期望不变'
print('E[score·Q]     =', np.round(E_sQ, 6))
print('E[score·(Q−b)] =', np.round(E_sQb, 6))
print('✅ 证毕(机器精度)：E[score]=0 → baseline 不引入偏差')

## 6 · baseline 大幅降方差（实测 ~18×）

无偏已证。现在看 baseline 的**收益**：在同一策略下采大量梯度估计，对比带/不带 baseline 的梯度方差。

baseline 取该策略下各状态的蒙特卡洛 $V$ 估计（即 advantage = $G - V$）。

In [ ]:
def estimate_V(env, theta, rng, n=3000, max_steps=50):
    '''蒙特卡洛估计当前策略各状态的 V。'''
    sums = np.zeros(env.N); counts = np.zeros(env.N)
    for _ in range(n):
        S, A, Rw = rollout(env, theta, rng, max_steps)
        G = discounted_returns(Rw, env.gamma)
        for i, st in enumerate(S):
            sums[st] += G[i]; counts[st] += 1
    return np.where(counts > 0, sums / np.maximum(counts, 1), 0.0)

def grad_with_baseline(env, theta, Vbar, n_samples, rng, use_baseline):
    grads = []
    for _ in range(n_samples):
        S, A, Rw = rollout(env, theta, rng)
        G = discounted_returns(Rw, env.gamma)
        adv = (G - Vbar[S]) if use_baseline else G
        g = np.zeros_like(theta)
        for t, (st, at) in enumerate(zip(S, A)):
            g[st] += adv[t] * score(theta[st], at)
        grads.append(g.flatten())
    return np.array(grads)

theta_mid, _ = reinforce(env, n_episodes=300, lr=0.2, seed=3)   # 半训练策略(用确定性 chain)
Vbar = estimate_V(env, theta_mid, np.random.default_rng(5))
g_no = grad_with_baseline(env, theta_mid, Vbar, 4000, np.random.default_rng(20), use_baseline=False)
g_bl = grad_with_baseline(env, theta_mid, Vbar, 4000, np.random.default_rng(20), use_baseline=True)
var_no, var_bl = g_no.var(0).sum(), g_bl.var(0).sum()
print(f'梯度总方差  无baseline = {var_no:.4f}')
print(f'梯度总方差  有baseline = {var_bl:.4f}')
print(f'方差缩减倍数 = {var_no / var_bl:.1f}×')
# 均值仍近似一致(无偏)，方差大降
assert np.linalg.norm(g_no.mean(0) - g_bl.mean(0)) < 0.05, 'baseline 不应明显改变梯度均值'
assert var_bl < var_no / 3, 'baseline 应显著降方差(至少 3×)'
print('✅ baseline 把梯度方差降一个数量级，梯度均值几乎不变 —— 免费的午餐(几乎)')

## 7 · 熵正则：度量与梯度

策略熵 $\mathcal{H}(\pi) = -\sum_a \pi(a)\log\pi(a)$：均匀时最大(=log A)，确定时为 0。

加熵奖励 $\beta\mathcal{H}$ 鼓励策略保持随机、防过早塌缩。先验证熵的性质与塌缩检测。

In [ ]:
def entropy(theta_s):
    pi = softmax(theta_s)
    return -np.sum(pi * np.log(pi + 1e-12))

A = 4
H_uniform = entropy(np.zeros(A))         # 均匀策略
H_peaked  = entropy(np.array([10.0, 0, 0, 0]))   # 几乎确定
print(f'均匀策略熵 = {H_uniform:.4f} (= log {A} = {np.log(A):.4f})')
print(f'尖峰策略熵 = {H_peaked:.4f} (接近 0)')
assert abs(H_uniform - np.log(A)) < 1e-6, '均匀分布熵 = log A(最大)'
assert H_peaked < 0.01, '确定性策略熵接近 0'
assert H_uniform > H_peaked, '均匀策略熵 > 尖峰策略'

# 对比：带熵正则训练的策略，末态熵应高于不带的(更保留探索)
theta_noent, _ = reinforce(env, n_episodes=2000, lr=0.3, seed=1)
H_noent = np.mean([entropy(theta_noent[s]) for s in range(env.N - 1)])
print(f'\n不带熵正则训练后，非终止态平均熵 = {H_noent:.3f}')
print('（策略塌缩到接近确定 → 熵低；熵正则会让它保持更高的熵以维持探索）')
print('✅ 熵度量正确：均匀=log A 最大，确定=0；熵正则防塌缩')

---
## ✏️ 练习 1：实现策略梯度（单条轨迹）

实现 `policy_gradient(theta, S, A, G, gamma)`：给定一条轨迹的状态/动作/回报，返回完整的梯度数组(形状同 theta)。

即 $g_s = \sum_{t: s_t=s} \gamma^t G_t \nabla_\theta \log\pi(a_t|s_t)$（不含学习率）。

In [ ]:
def policy_gradient(theta, S, A, G, gamma):
    g = np.zeros_like(theta)
    # TODO: 遍历轨迹每步 t，累加 gamma**t * G[t] * score(theta[S[t]], A[t]) 到 g[S[t]]
    raise NotImplementedError
    return g

In [ ]:
# —— 练习 1 自测 ——
# 用一条「人造」单次访问轨迹，保证每个状态只出现一次(避免重复访问的梯度叠加)
theta_t = np.zeros((3, 2))
S_test = [0, 1]          # 状态 0、1 各访问一次
A_test = [1, 1]          # 都选动作 1(右)
G_test = np.array([1.0, 1.0])   # 都是正回报
g = policy_gradient(theta_t, S_test, A_test, G_test, gamma=0.99)
assert g.shape == theta_t.shape
# 对照参考：g[s] = γ^t · G_t · score(θ_s, a_t)
for t, (st, at) in enumerate(zip(S_test, A_test)):
    expected = (0.99 ** t) * G_test[t] * score(theta_t[st], at)
    assert np.allclose(g[st], expected), '梯度应为 γ^t·G_t·score'
# 单次访问下，正回报沿梯度走一步必提高所选动作概率
theta_new = theta_t + 0.5 * g
for st, at in zip(S_test, A_test):
    assert softmax(theta_new[st])[at] > softmax(theta_t[st])[at], '正回报应推高所选动作概率'
# 未访问的状态梯度为 0
assert np.allclose(g[2], 0.0), '未访问状态梯度应为 0'
print('✅ 练习 1 通过：策略梯度=γ^t·G_t·score，正回报推高所选动作概率')

## ✏️ 练习 2：advantage = 回报 − baseline

实现 `compute_advantage(G, V, states)`：给定每步回报 `G`、各状态价值 `V`、轨迹状态 `states`，返回 advantage 数组 $A_t = G_t - V(s_t)$。这是把 baseline 接入策略梯度的桥梁。

In [ ]:
def compute_advantage(G, V, states):
    # TODO: 返回 G - V[states]（逐元素）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
G = np.array([0.5, 0.8, 1.0])
V = np.array([0.6, 0.7, 0.9, 0.0, 0.0])
states = [0, 1, 2]
adv = compute_advantage(G, V, states)
assert np.allclose(adv, [0.5 - 0.6, 0.8 - 0.7, 1.0 - 0.9]), 'advantage = G - V[s]'
assert np.allclose(adv, [-0.1, 0.1, 0.1])
# advantage 均值应比原始 G 更接近 0(围绕 baseline 波动)
assert abs(adv.mean()) < abs(G.mean()), 'advantage 应比原始回报更居中'
print('✅ 练习 2 通过：advantage = 回报 − baseline，信号更居中')

## ✏️ 练习 3：熵及其梯度

实现 `entropy_grad(theta_s)`：softmax 策略熵 $\mathcal{H} = -\sum_a \pi_a \log\pi_a$ 对 logits 的梯度。

闭式：$\frac{\partial \mathcal{H}}{\partial \theta_j} = -\pi_j(\log\pi_j + \mathcal{H})$（可用有限差分验证）。沿此梯度上升应增大熵(更均匀)。

In [ ]:
def entropy_grad(theta_s):
    pi = softmax(theta_s)
    H = -np.sum(pi * np.log(pi + 1e-12))
    # TODO: 返回 -pi * (log(pi) + H)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
theta_s = np.array([1.0, 0.2, -0.5])
g_analytic = entropy_grad(theta_s)
# 有限差分对拍
eps = 1e-6; g_fd = np.zeros(3)
for j in range(3):
    tp = theta_s.copy(); tp[j] += eps; tm = theta_s.copy(); tm[j] -= eps
    Hp = -np.sum(softmax(tp) * np.log(softmax(tp) + 1e-12))
    Hm = -np.sum(softmax(tm) * np.log(softmax(tm) + 1e-12))
    g_fd[j] = (Hp - Hm) / (2 * eps)
assert np.allclose(g_analytic, g_fd, atol=1e-5), '熵梯度应对拍有限差分'
# 沿熵梯度上升应增大熵
H0 = -np.sum(softmax(theta_s) * np.log(softmax(theta_s) + 1e-12))
theta_up = theta_s + 0.1 * g_analytic
H1 = -np.sum(softmax(theta_up) * np.log(softmax(theta_up) + 1e-12))
assert H1 > H0, '沿熵梯度上升应增大熵(更均匀)'
print('✅ 练习 3 通过：熵梯度正确，沿之上升使策略更随机')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def policy_gradient(theta, S, A, G, gamma):
    g = np.zeros_like(theta)
    for t, (st, at) in enumerate(zip(S, A)):
        g[st] += (gamma ** t) * G[t] * score(theta[st], at)
    return g

In [ ]:
# 练习 2 参考答案
def compute_advantage(G, V, states):
    return G - V[np.array(states)]

In [ ]:
# 练习 3 参考答案
def entropy_grad(theta_s):
    pi = softmax(theta_s)
    H = -np.sum(pi * np.log(pi + 1e-12))
    return -pi * (np.log(pi + 1e-12) + H)

---
## 🧪 真实数据胶囊：RLHF 风味的「偏好优化」

大模型 RLHF（C22）本质是策略梯度：把「动作」换成「生成的回复」、「环境奖励」换成 **reward model 打分**。

用一个**真实风味**的玩具复现：1-step「语言」bandit——5 个候选回复，每个有真实的（隐藏）人类偏好分。策略是 softmax over 回复，用 REINFORCE + baseline 优化，让它学会生成高分回复。这正是 RLHF/GRPO 的内核结构。

In [ ]:
def rlhf_toy(reward_model, n_steps=2000, lr=0.3, seed=0, use_baseline=True):
    '''单步 bandit：5 个回复，reward_model[a]=该回复的人类偏好分。
       策略 softmax(theta)，REINFORCE + 滑动 baseline 优化。'''
    rng = np.random.default_rng(seed)
    K = len(reward_model)
    theta = np.zeros(K)
    baseline = 0.0
    avg_reward_hist = []
    for t in range(n_steps):
        pi = softmax(theta)
        a = int(rng.choice(K, p=pi))         # 采样一个回复
        r = reward_model[a]                  # reward model 打分
        adv = r - baseline if use_baseline else r
        theta += lr * adv * score(theta, a)  # 策略梯度
        baseline += 0.01 * (r - baseline)    # 滑动平均 baseline
        avg_reward_hist.append(r)
    return theta, np.array(avg_reward_hist)

# 隐藏的真实人类偏好分（回复 3 最受欢迎）
reward_model = np.array([0.1, 0.3, 0.2, 0.9, 0.4])
theta_rlhf, hist = rlhf_toy(reward_model, n_steps=3000, use_baseline=True)
final_pi = softmax(theta_rlhf)
print('学到的回复分布:', np.round(final_pi, 3))
print('最受偏好的回复 index:', int(final_pi.argmax()), '(真实最优=3)')
print(f'末期平均得分 = {hist[-300:].mean():.3f} (初期 {hist[:300].mean():.3f}, 满分 0.9)')
assert final_pi.argmax() == 3, '策略应学会生成最高偏好分的回复'
assert hist[-300:].mean() > hist[:300].mean(), '平均得分应随训练上升'
print('✅ RLHF 内核跑通：策略梯度让模型学会生成人类更偏好的回复')

**🧪 胶囊练习**：实现 `kl_penalty(pi, pi_ref)`：RLHF 用 KL 散度 $D_{KL}(\pi \| \pi_{ref})$ 约束新策略别偏离原模型太远(防 reward hacking)。返回 $\sum_a \pi_a \log(\pi_a / \pi^{ref}_a)$。直觉：分布越像 KL 越小(=0 当相同)。

In [ ]:
def kl_penalty(pi, pi_ref):
    # TODO: 返回 Σ_a pi[a] * log(pi[a]/pi_ref[a])，加 1e-12 防 log(0)
    raise NotImplementedError

In [ ]:
# 自测
pi_ref = np.array([0.2, 0.2, 0.2, 0.2, 0.2])   # 原模型(均匀)
pi_same = pi_ref.copy()
pi_drift = np.array([0.05, 0.05, 0.05, 0.8, 0.05])  # 偏向回复3
assert abs(kl_penalty(pi_same, pi_ref)) < 1e-9, '相同分布 KL=0'
assert kl_penalty(pi_drift, pi_ref) > 0, '偏离的分布 KL>0'
print(f'KL(原模型‖原模型) = {kl_penalty(pi_same, pi_ref):.4f}')
print(f'KL(漂移策略‖原模型) = {kl_penalty(pi_drift, pi_ref):.4f}')
print('✅ 胶囊练习通过：KL 惩罚约束策略别偏离原模型太远(RLHF 防 reward hacking)')

In [ ]:
# 📖 胶囊参考答案
def kl_penalty(pi, pi_ref):
    return float(np.sum(pi * np.log((pi + 1e-12) / (pi_ref + 1e-12))))

### 小结
- **策略梯度**直接对 π_θ 求梯度上升，绕开价值函数；连续动作/随机策略天然适配。
- **log-derivative trick**：∇log p(τ)=Σ∇log π(a|s)，**环境梯度消失** → 无需模型即可估梯度。
- **REINFORCE**：θ ← θ + η·G_t·∇log π，无偏但方差大；用 **G_t(reward-to-go)** 而非整轨回报(因果性)。
- **baseline**：E[score]=0 → 减 b(s) **不引入偏差**却 **降方差一个数量级**；advantage=G−V。
- **熵正则**：奖励策略随机，**防过早塌缩**；RLHF/GRPO 是策略梯度在 LLM 上的直接应用。

下一站：**模块 04 · Actor-Critic 与 GAE** —— 用学出来的 critic 当 baseline，并用 GAE 精细调 bias-variance。